<a href="https://colab.research.google.com/github/M4rck0/Datos_Masivos/blob/main/Tarea_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejecutar Spark en Google Colab

In [1]:
%%capture
!pip -q uninstall -y dataproc-spark-connect pyspark py4j # Limpieza e instalación
!pip -q install --no-cache-dir pyspark==3.5.1

# Instalar Java
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq


In [2]:
import os
from pyspark.sql import SparkSession
from datasets import load_dataset
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Variables de entorno
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"

!java -version

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


In [3]:
spark = (
    SparkSession.builder
    .appName("SparkEnColab")
    .master("local[*]")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

print("Version de spark:", spark.version)

Version de spark: 3.5.1


## Prueba de funcionamiento

In [4]:
df_prueba = spark.createDataFrame([(1,"a"), (2,"b"), (3,"c")], ["id","valor"])
df_prueba.show()

+---+-----+
| id|valor|
+---+-----+
|  1|    a|
|  2|    b|
|  3|    c|
+---+-----+



In [5]:
df_prueba.createOrReplaceTempView("t")
spark.sql("SELECT valor, COUNT(*) AS n FROM t GROUP BY valor").show()

+-----+---+
|valor|  n|
+-----+---+
|    a|  1|
|    c|  1|
|    b|  1|
+-----+---+



# Dataset elegido y justificación

**Dataset elegido:** `HuggingFaceGECLM/REDDIT_comments` (partición: `technology`).

**Descripción:** Conjunto de comentarios de Reddit que incluye campos como:
- `created_utc` (timestamp),
- `author` (usuario),
- `score` (puntuación del comentario),
- `body` (texto del comentario).

**Justificación:** Se eligió porque es un dataset real y grande, ideal para practicar metadatos con operaciones de PySpark como limpieza, filtrado, agregaciones por tiempo y estadísticas descriptivas, además para practicar con PySpark.

# Carga del dataset con PySpark

In [6]:
# Parámetros
subreddit = "technology" # Subreddit a descargar
filas_maximas = 200_000 # Límite total de filas a descargar
tam_lote = 50000 # Tamaño del lote
lote = [] # Acumular filas hasta llegar al tamaño del lote
n = 0 # Contador total de filas procesadas
parte = 0 # Número de carpeta que se guarda

# Esquema
esquema = StructType([
    StructField("subreddit", StringType(), True),
    StructField("created_utc", LongType(), True),
    StructField("author", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("body", StringType(), True),
])

# Convierte a int, sino regresa none
def to_int(x):
    try:
        if x is None:
            return None
        return int(x)
    except Exception:
        return None

# Cargar dataset en modo streaming
# streaming = true significa que no se descarga todo a ram
# En su lugar, se itera registro por registro
ds = load_dataset(
    "HuggingFaceGECLM/REDDIT_comments",
    split=subreddit,
    streaming=True
)

for fila in ds:
    # Convertimos cada registro a una tupla con el orden del esquema
    lote.append((
        fila.get("subreddit"),
        to_int(fila.get("created_utc")),
        fila.get("author"),
        to_int(fila.get("score")),
        fila.get("body"),
    ))
    n += 1

    # Cada vez que juntamos filas:
    # 1) Creamos un dataframe de spark con esquema fijo
    # 2) Lo escribimos en parquet usando spark
    # 3) Limpiamos el lote y avanzamos el contador de partes
    if n % tam_lote == 0:
        df_lote = spark.createDataFrame(lote, schema=esquema)

        # Spark escribe a carpetas
        ruta_salida = f"/content/reddit_spark/{subreddit}/parte{parte:03d}"
        df_lote.write.mode("overwrite").parquet(ruta_salida)

        lote = []
        parte += 1
        print(f"Guardado: {ruta_salida} | filas acumuladas: {n}")

    # Terminar si llegamos al límite
    if n >= filas_maximas:
        break


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/45 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/46 [00:00<?, ?it/s]

Guardado: /content/reddit_spark/technology/parte000 | filas acumuladas: 50000
Guardado: /content/reddit_spark/technology/parte001 | filas acumuladas: 100000
Guardado: /content/reddit_spark/technology/parte002 | filas acumuladas: 150000
Guardado: /content/reddit_spark/technology/parte003 | filas acumuladas: 200000


In [7]:
df_completo = (spark.read
          .option("recursiveFileLookup", "true")
          .parquet(f"/content/reddit_spark/{subreddit}")
          .withColumn("subreddit", F.lit(subreddit))
         )

print("Filas leídas:", df_completo.count())

# Comentarios no borrados
(df_completo
 .filter((F.col("body").isNotNull()) & (F.col("body") != "[deleted]"))
 .select("created_utc", "author", "score", "body")
 .show(10, truncate=120))

Filas leídas: 200000
+-----------+-------------+-----+------------------------------------------------------------------------------------------------------------------------+
|created_utc|       author|score|                                                                                                                    body|
+-----------+-------------+-----+------------------------------------------------------------------------------------------------------------------------+
| 1432313203|      AbeRego|    3|                                                                         I read this in the cliché teen Simpson's voice.|
| 1432313210|   hefnetefne|    1|How about a law that says you can sue corporations, instead of proclaiming corporations people so you can sue them? \...|
| 1432313243|newloginisnew|   23|Please share 100% of your browsing history as a comment reply. If you do not, then you obviously have something to hide.|
| 1432313244|       NeonHD|    1|My post wasn't s

# PySpark: filtros, estadísticas descriptivas y operaciones aritméticas

In [15]:
# Preparación: timestamp y fecha (en el dataset aparece null)
df = (df_completo
      .withColumn("subreddit", F.lit(subreddit))
      .withColumn("created_ts", F.to_timestamp(F.from_unixtime("created_utc")))
      .withColumn("date", F.to_date("created_ts"))
     )

# Filtrar comentarios no borrados
df_filtrado = (df
      .filter(F.col("author").isNotNull())
      .filter(F.col("author") != "[deleted]")
      .filter(F.col("body").isNotNull())
      .filter(F.col("body") != "[deleted]")
      .filter(F.col("score").isNotNull())
)

print("Total original:", df.count())
print("Total validos:", df_filtrado.count())

# score >= 10
df_score = df_filtrado.filter(F.col("score") >= 10)
print("Comentarios con score >= 10:", df_score.count())
df_score.select("date", "author", "score", "body").orderBy(F.desc("score")).show(5, truncate=80)

# Estadísticas descriptivas
df_filtrado.select("score").describe().show()

# Estadísticas
estadisticas = (df_filtrado
    .groupBy("date")
    .agg(
        F.count("*").alias("comentarios"),
        F.avg("score").alias("promedio_score"),
        F.max("score").alias("max_score")
    )
    .orderBy("date")
)
estadisticas.show(10)

# Operaciones aritméticas
df_oparit = df_filtrado.withColumn("longitud_texto", F.length("body"))

df_oparit = df_oparit.withColumn(
    "puntaje_x_100_caracteres",
    F.when(F.col("longitud_texto") > 0,
           (F.col("score") / F.col("longitud_texto")) * 100
    ).otherwise(None)
)

df_oparit.select("author", "score", "longitud_texto", "puntaje_x_100_caracteres").show(5, truncate=80)

# Operación entre registros: diferencia del conteo diario vs día anterior (lag)
w = Window.orderBy("date")

dif_diaria = (estadisticas
    .withColumn("comentarios_previos", F.lag("comentarios").over(w))
    .withColumn("dif_vs_dia_anterior", F.col("comentarios") - F.col("comentarios_previos"))
    .withColumn(
        "porc_cambio_vs_dia_anterior",
        F.when(F.col("comentarios_previos").isNull(), None)
         .otherwise((F.col("dif_vs_dia_anterior") / F.col("comentarios_previos")) * 100)
    )
)

dif_diaria.show(10)

Total original: 200000
Total validos: 180078
Comentarios con score >= 10: 25325
+----------+-------------+-----+--------------------------------------------------------------------------------+
|      date|       author|score|                                                                            body|
+----------+-------------+-----+--------------------------------------------------------------------------------+
|2018-12-18|BrinnerTechie|23281|&gt; "Officer Rivas then accused Mr. Elsharkawi of hiding something because o...|
|2018-12-02|    DWMoose83|20837|              No, it's not. Which is why healthcare shouldn't be about business.|
|2018-12-12|    Ranvier01|13986|The corrupt:\n\nBrendan Boyle (PA-13) - Comcast, Verizon, NCTA\n\nRobert Brad...|
|2018-12-03|     limbodog|12180|                                      December 18th sees Tumblr become obsolete.|
|2018-12-21|   phydeaux70|10856|You are quitting Facebook if you quit Facebook.\n\nYou aren't hurting Faceboo...|
+-------